In [8]:
#!pip install transformers
#!pip install sentencepiece tiktoken
#!pip install llama-cpp-python

In [2]:
import os #Environment variables
from gpt4all import GPT4All
from transformers import AutoTokenizer

In [3]:
# Load Model pointing using GPU
model = GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf", device="cuda") 

In [25]:
### Get Model Source
print('model name: {} \n'.format(model.config['name']),
      'model url: {}'.format(model.config['url']),
      )

model name: Llama 3.2 3B Instruct 
 model url: https://huggingface.co/bartowski/Llama-3.2-3B-Instruct-GGUF/resolve/main/Llama-3.2-3B-Instruct-Q4_0.gguf


In [29]:
model.close() #close model

#### Sources

- https://docs.gpt4all.io/gpt4all_desktop/chat_templates.html#errorwarning-system-message-is-not-plain-text
- https://ranjankumar.in/chat-templates-the-last-unowned-layer-in-your-llm-stack
- https://labuladong.online/en/ai-coding/basics/hugging-face-model-files/

#### From model repos
- Download tokenizers from repos such as Hugging Face.
- Does not guarantee same behavior as tokenizer used by GPT4All backend when calling the model; specially for gated models such as Llama or Mistral.
- For the case of gated models (where public access to their chat template and tokenizer is allowed), follow this workflow:
      - Find a community developed clone (you can check the model name and its url with the GPT4All model instance, see above).
      - Find the link to its non-GGUF (i.e. non binary form), usually found in its README.
      - If the non-GGUF has a tokenizer.json, use this Hugging Face url (see below). Otherwise (such as tokenizer.ggml files, which means the tokenizer is embedded in the GGUF, see below), find another community developed clone that can be equivalent for using with Auto tokenizer (behavior might not replicate exactly the behavior of the tokenizer used under the hood by the GPT4All backend). 

In [27]:
# Carga el tokenizador oficial de la arquitectura Llama 3
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct")

In [28]:
# Tu chat formateado
prompt_renderizado = "<|start_header_id|>user<|end_header_id|>\n\nHola, ¿cómo estás?<|eot_id|>"
token_ids = tokenizer.encode(prompt_renderizado)

print("IDs de los Tokens:", token_ids)
print("N° tokens:", len(token_ids))

IDs de los Tokens: [128000, 128006, 882, 128007, 271, 69112, 11, 29386, 66, 72561, 1826, 7206, 30, 128009]
N° tokens: 14


#### Local Model file (GGUF
- From local model file downloaded from repos such as Hugging Face.
- Uses llama.cpp binding to read the tokenizer .json packed with model file binary (GGUF).
- Best way to replicate the tokenizer behavior as executed by GPT4All backend (which binds to llama.cpp).

In [4]:
from llama_cpp import Llama

In [ ]:
path="path_to_models"

In [7]:
model_file="Llama-3.2-3B-Instruct-Q4_0.gguf" #model file

In [9]:
# Cargamos el modelo (puedes activar verbose=False para limpiar la consola)
# Pasamos embedding=True para que consuma el mínimo de memoria posible
llm = Llama(model_path=path+model_file, vocab_only=True, verbose=False)

llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


In [14]:
texto = "Hola, cómo estás? Replicando el tokenizador."

# Codificar (Convertir texto a IDs)
token_ids = llm.tokenize(texto.encode('utf-8'))
print("IDs de los Tokens:", token_ids)
print("N° tokens:", len(token_ids))

IDs de los Tokens: [128000, 69112, 11, 55996, 1826, 7206, 30, 3402, 416, 4988, 658, 4037, 91689, 13]
N° tokens: 14


In [15]:
llm.close() #close the model client